# CoachPeaking TRIMP HistGradientBoosting Pilot

This notebook compares a histogram-based gradient boosting model with the first Gradient Boosting pilot. It uses exactly the same reviewed dataset, features, and chronological train/development split so that the development metrics are directly comparable.

## Feature set

The feature set is: duration, average speed, average running cadence, and heart-rate zone shares 2 to 5. Zone 1 is omitted because the five zone shares sum to one. Distance, speed maxima, average/max heart rate, Garmin training load, and training effects are excluded from this first comparison to avoid obvious redundancy or derived-metric leakage.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATASET_PATH = Path('data/coachpeaking-trimp-dataset/2025/working/TRIMP_TRAIN_REVIEW.csv')
if not DATASET_PATH.exists():
    DATASET_PATH = Path('..') / DATASET_PATH
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Review dataset not found: {DATASET_PATH}')


In [ ]:
data = pd.read_csv(DATASET_PATH, parse_dates=['ACTIVITY_START_DATE'])
data = data.sort_values('ACTIVITY_START_DATE').reset_index(drop=True)
target_column = 'COACHPEAKING_TRIMP'
if data[target_column].isna().any():
    raise ValueError('The review dataset still contains missing TRIMP labels.')

for zone in range(1, 6):
    data[f'HR_ZONE_{zone}_SHARE'] = data[f'HR_TIME_IN_ZONE_{zone}'] / data['DURATION_SECONDS']

features = [
    'DURATION_SECONDS', 'AVERAGE_SPEED_MPS', 'AVERAGE_RUNNING_CADENCE_SPM',
    'HR_ZONE_2_SHARE', 'HR_ZONE_3_SHARE', 'HR_ZONE_4_SHARE', 'HR_ZONE_5_SHARE',
]
if data[features].isna().any().any():
    raise ValueError(f'Missing values in pilot features: {data[features].columns[data[features].isna().any()].tolist()}')


## Sequential train/development split

The earliest 80% of activities is the training set. The latest 20% is the development set. No random shuffle is used.

In [ ]:
split_index = int(len(data) * 0.80)
train_data, dev_data = data.iloc[:split_index].copy(), data.iloc[split_index:].copy()
X_train, y_train = train_data[features], train_data[target_column]
X_dev, y_dev = dev_data[features], dev_data[target_column]
print(f'Train: {len(train_data)} rows, {train_data.ACTIVITY_START_DATE.min().date()} to {train_data.ACTIVITY_START_DATE.max().date()}')
print(f'Dev:   {len(dev_data)} rows, {dev_data.ACTIVITY_START_DATE.min().date()} to {dev_data.ACTIVITY_START_DATE.max().date()}')


## HistGradientBoosting training and development evaluation

The model is kept shallow and regularised because the training set is small. It is evaluated only on the future development partition.

In [ ]:
model = HistGradientBoostingRegressor(
    learning_rate=0.05, max_iter=100, max_leaf_nodes=7,
    min_samples_leaf=10, l2_regularization=1.0, random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
predictions = model.predict(X_dev)
metrics = pd.Series({
    'MAE': mean_absolute_error(y_dev, predictions),
    'RMSE': mean_squared_error(y_dev, predictions) ** 0.5,
    'MAPE_percent': ((y_dev - predictions).abs() / y_dev).mean() * 100,
    'R2': r2_score(y_dev, predictions),
}).round(3)
metrics


In [ ]:
evaluation = dev_data[['ACTIVITY_START_DATE', target_column]].copy()
evaluation['PREDICTED_TRIMP'] = predictions
evaluation['ABSOLUTE_ERROR'] = (evaluation[target_column] - evaluation['PREDICTED_TRIMP']).abs()
display(evaluation)

importance = permutation_importance(
    model, X_dev, y_dev, scoring='neg_mean_absolute_error',
    n_repeats=30, random_state=RANDOM_STATE,
).importances_mean
importance = pd.Series(importance, index=features).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(y_dev, predictions)
limits = [min(y_dev.min(), predictions.min()), max(y_dev.max(), predictions.max())]
axes[0].plot(limits, limits, 'k--', linewidth=1)
axes[0].set(xlabel='Observed TRIMP', ylabel='Predicted TRIMP', title='Development predictions')
importance.plot.barh(ax=axes[1], title='Development permutation importance')
axes[1].set_xlabel('Importance for negative MAE')
plt.tight_layout()
